In [ ]:
import pandas as pd

In [ ]:
matches = pd.read_csv('matches.csv')
matches.shape

(1389, 28)

In [ ]:
##38 matches*20 teams*2 seasons = 1520
matches["team"].value_counts()

,count
team,
Manchester United,72
West Ham United,72
Newcastle United,72
Brighton and Hove Albion,72
Southampton,72
Tottenham Hotspur,71
Manchester City,71
Leeds United,71
Wolverhampton Wanderers,71


In [ ]:
matches["round"].value_counts()

,count
round,
Matchweek 1,39
Matchweek 2,39
Matchweek 3,39
Matchweek 4,39
Matchweek 5,39
Matchweek 6,39
Matchweek 7,39
Matchweek 8,39
Matchweek 9,39


In [ ]:
matches.dtypes

,0
Unnamed: 0,int64
date,object
time,object
comp,object
round,object
day,object
venue,object
result,object
gf,float64
ga,float64


In [ ]:
matches["date"] = pd.to_datetime(matches["date"])

/tmp/ipykernel_807/2174778740.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches["date"] = pd.to_datetime(matches["date"])


In [ ]:
matches.dtypes

,0
date,datetime64[ns]
time,object
comp,object
round,object
day,object
venue,object
result,object
gf,float64
ga,float64
opponent,object


In [ ]:
# 0 : Away
#  1:Home
matches["venue_code"]= matches["venue"].astype("category").cat.codes

In [ ]:
matches["opp_code"] = matches["opponent"].astype("category").cat.codes

In [ ]:
matches["hour"] = matches["time"].str.split(":").str[0].astype("float").astype("int64")

In [ ]:
matches["day_code"] = matches["date"].dt.dayofweek

In [ ]:
matches = matches.copy()
matches["target"] = (matches["result"] == "W").astype("int")

In [ ]:
matches = matches.loc[:, ~matches.columns.str.contains('^Unnamed')]

In [ ]:
matches.head()

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,fk,pk,pkatt,season,team,venue_code,opp_code,hour,day_code,target
0,2021-08-15,16:30,Premier League,Matchweek 1,Sun,Away,L,0.0,1.0,Tottenham,...,1.0,0.0,0.0,2022,Manchester City,0,18,16,6,0
1,2021-08-21,15:00,Premier League,Matchweek 2,Sat,Home,W,5.0,0.0,Norwich City,...,1.0,0.0,0.0,2022,Manchester City,1,15,15,5,0
2,2021-08-28,12:30,Premier League,Matchweek 3,Sat,Home,W,5.0,0.0,Arsenal,...,0.0,0.0,0.0,2022,Manchester City,1,0,12,5,0
3,2021-09-11,15:00,Premier League,Matchweek 4,Sat,Away,W,1.0,0.0,Leicester City,...,0.0,0.0,0.0,2022,Manchester City,0,10,15,5,0
4,2021-09-18,15:00,Premier League,Matchweek 5,Sat,Home,D,0.0,0.0,Southampton,...,1.0,0.0,0.0,2022,Manchester City,1,17,15,5,0


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

In [ ]:
train = matches[matches["date"] < '2022-01-01']
test = matches[matches["date"] >= '2022-01-01']

In [ ]:
predictors = ["venue_code", "opp_code", "hour", "day_code"]

In [ ]:
rf.fit(train[predictors],train["target"])

RandomForestClassifier(min_samples_split=10, n_estimators=50, random_state=1)

In [ ]:
preds = rf.predict(test[predictors])

In [ ]:
from sklearn.metrics import accuracy_score
acc = accuracy_score(test["target"], preds)

In [ ]:
acc

0.6134751773049646

In [ ]:
combined = pd.DataFrame(dict(actual=test["target"], predicted=preds))

In [ ]:
pd.crosstab(index=combined["actual"],columns=combined['predicted'])

predicted,0,1
actual,,
0,144,31
1,78,29


In [ ]:
from sklearn.metrics import precision_score
precision_score(test["target"], preds)

0.48333333333333334

In [ ]:
grouped_matches = matches.groupby("team")
group = grouped_matches.get_group("Arsenal")

In [ ]:
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [ ]:
cols = ["gf","ga","sh","sot","dist","fk","pk","pkatt"]
new_cols = [f"{c}_rolling" for c in cols]

In [ ]:
new_cols

['gf_rolling',
 'ga_rolling',
 'sh_rolling',
 'sot_rolling',
 'dist_rolling',
 'fk_rolling',
 'pk_rolling',
 'pkatt_rolling']

In [ ]:
rolling_averages(group, cols, new_cols)

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,day_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling,dist_rolling,fk_rolling,pk_rolling,pkatt_rolling
898,2020-10-04,14:00,Premier League,Matchweek 4,Sun,Home,W,2.0,1.0,Sheffield Utd,...,6,1,2.000000,1.333333,7.666667,3.666667,14.733333,0.666667,0.000000,0.000000
899,2020-10-17,17:30,Premier League,Matchweek 5,Sat,Away,L,0.0,1.0,Manchester City,...,5,0,1.666667,1.666667,5.333333,3.666667,15.766667,0.000000,0.000000,0.000000
900,2020-10-25,19:15,Premier League,Matchweek 6,Sun,Home,L,0.0,1.0,Leicester City,...,6,0,1.000000,1.666667,7.000000,3.666667,16.733333,0.666667,0.000000,0.000000
901,2020-11-01,16:30,Premier League,Matchweek 7,Sun,Away,W,1.0,0.0,Manchester Utd,...,6,1,0.666667,1.000000,9.666667,4.000000,16.033333,1.000000,0.000000,0.000000
902,2020-11-08,19:15,Premier League,Matchweek 8,Sun,Home,L,0.0,3.0,Aston Villa,...,6,0,0.333333,0.666667,9.666667,2.666667,18.033333,1.000000,0.333333,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,2022-04-04,20:00,Premier League,Matchweek 31,Mon,Away,L,0.0,3.0,Crystal Palace,...,0,0,1.000000,0.666667,13.000000,3.666667,18.400000,0.000000,0.333333,0.333333
94,2022-04-09,15:00,Premier League,Matchweek 32,Sat,Home,L,1.0,2.0,Brighton,...,5,0,0.333333,1.666667,10.333333,2.666667,19.300000,0.000000,0.000000,0.000000
95,2022-04-16,15:00,Premier League,Matchweek 33,Sat,Away,L,0.0,1.0,Southampton,...,5,0,0.666667,1.666667,14.000000,3.333333,17.566667,0.666667,0.000000,0.000000
96,2022-04-20,19:45,Premier League,Matchweek 25,Wed,Away,W,4.0,2.0,Chelsea,...,2,1,0.333333,2.000000,18.333333,4.333333,17.400000,1.000000,0.000000,0.000000


In [ ]:
matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))

/tmp/ipykernel_807/4281467257.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))


In [ ]:
matches_rolling

date   time            comp         round  \
team                                                                          
Arsenal                 898 2020-10-04  14:00  Premier League   Matchweek 4   
                        899 2020-10-17  17:30  Premier League   Matchweek 5   
                        900 2020-10-25  19:15  Premier League   Matchweek 6   
                        901 2020-11-01  16:30  Premier League   Matchweek 7   
                        902 2020-11-08  19:15  Premier League   Matchweek 8   
...                                ...    ...             ...           ...   
Wolverhampton Wanderers 227 2022-03-13  14:00  Premier League  Matchweek 29   
                        228 2022-03-18  20:00  Premier League  Matchweek 30   
                        229 2022-04-02  15:00  Premier League  Matchweek 31   
                        230 2022-04-08  20:00  Premier League  Matchweek 32   
                        231 2022-04-24  14:00  Premier League  Matchweek 34   

                             day venue result   gf   ga         opponent  ...  \
team                                                                      ...   
Arsenal                 898  Sun  Home      W  2.0  1.0    Sheffield Utd  ...   
                        899  Sat  Away      L  0.0  1.0  Manchester City  ...   
                        900  Sun  Home      L  0.0  1.0   Leicester City  ...   
                        901  Sun  Away      W  1.0  0.0   Manchester Utd  ...   
                        902  Sun  Home      L  0.0  3.0      Aston Villa  ...   
...                          ...   ...    ...  ...  ...              ...  ...   
Wolverhampton Wanderers 227  Sun  Away      W  1.0  0.0          Everton  ...   
                        228  Fri  Home      L  2.0  3.0     Leeds United  ...   
                        229  Sat  Home      W  2.0  1.0      Aston Villa  ...   
                        230  Fri  Away      L  0.0  1.0    Newcastle Utd  ...   
                        231  Sun  Away      L  0.0  1.0          Burnley  ...   

                             day_code  target  gf_rolling  ga_rolling  \
team                                                                    
Arsenal                 898         6       1    2.000000    1.333333   
                        899         5       0    1.666667    1.666667   
                        900         6       0    1.000000    1.666667   
                        901         6       1    0.666667    1.000000   
                        902         6       0    0.333333    0.666667   
...                               ...     ...         ...         ...   
Wolverhampton Wanderers 227         6       1    1.333333    1.000000   
                        228         4       0    1.666667    0.666667   
                        229         5       1    2.333333    1.000000   
                        230         4       0    1.666667    1.333333   
                        231         6       0    1.333333    1.666667   

                            sh_rolling sot_rolling dist_rolling fk_rolling  \
team                                                                         
Arsenal                 898   7.666667    3.666667    14.733333   0.666667   
                        899   5.333333    3.666667    15.766667   0.000000   
                        900   7.000000    3.666667    16.733333   0.666667   
                        901   9.666667    4.000000    16.033333   1.000000   
                        902   9.666667    2.666667    18.033333   1.000000   
...                                ...         ...          ...        ...   
Wolverhampton Wanderers 227  12.333333    3.666667    19.300000   0.000000   
                        228  12.333333    4.333333    19.600000   0.000000   
                        229  13.000000    5.333333    19.833333   0.000000   
                        230  13.000000    5.000000    18.533333   0.000000   
                        231  10.000000    4.666667    17.633333   

In [ ]:
matches_rolling = matches_rolling.droplevel('team')

In [ ]:
matches_rolling.index = range(matches_rolling.shape[0])

In [ ]:
matches_rolling

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,day_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling,dist_rolling,fk_rolling,pk_rolling,pkatt_rolling
0,2020-10-04,14:00,Premier League,Matchweek 4,Sun,Home,W,2.0,1.0,Sheffield Utd,...,6,1,2.000000,1.333333,7.666667,3.666667,14.733333,0.666667,0.000000,0.000000
1,2020-10-17,17:30,Premier League,Matchweek 5,Sat,Away,L,0.0,1.0,Manchester City,...,5,0,1.666667,1.666667,5.333333,3.666667,15.766667,0.000000,0.000000,0.000000
2,2020-10-25,19:15,Premier League,Matchweek 6,Sun,Home,L,0.0,1.0,Leicester City,...,6,0,1.000000,1.666667,7.000000,3.666667,16.733333,0.666667,0.000000,0.000000
3,2020-11-01,16:30,Premier League,Matchweek 7,Sun,Away,W,1.0,0.0,Manchester Utd,...,6,1,0.666667,1.000000,9.666667,4.000000,16.033333,1.000000,0.000000,0.000000
4,2020-11-08,19:15,Premier League,Matchweek 8,Sun,Home,L,0.0,3.0,Aston Villa,...,6,0,0.333333,0.666667,9.666667,2.666667,18.033333,1.000000,0.333333,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1312,2022-03-13,14:00,Premier League,Matchweek 29,Sun,Away,W,1.0,0.0,Everton,...,6,1,1.333333,1.000000,12.333333,3.666667,19.300000,0.000000,0.000000,0.000000
1313,2022-03-18,20:00,Premier League,Matchweek 30,Fri,Home,L,2.0,3.0,Leeds United,...,4,0,1.666667,0.666667,12.333333,4.333333,19.600000,0.000000,0.000000,0.000000
1314,2022-04-02,15:00,Premier League,Matchweek 31,Sat,Home,W,2.0,1.0,Aston Villa,...,5,1,2.333333,1.000000,13.000000,5.333333,19.833333,0.000000,0.000000,0.000000
1315,2022-04-08,20:00,Premier League,Matchweek 32,Fri,Away,L,0.0,1.0,Newcastle Utd,...,4,0,1.666667,1.333333,13.000000,5.000000,18.533333,0.000000,0.000000,0.000000


In [ ]:
def make_predictions(data, predictors):
  train = data[data["date"] < '2022-01-01']
  test = data[data["date"] >= '2022-01-01']
  rf.fit(train[predictors], train["target"])
  preds = rf.predict(test[predictors])
  combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
  precision = precision_score(test["target"], preds)
  return combined, precision

In [ ]:
combined, precision = make_predictions(matches_rolling, predictors + new_cols)

In [ ]:
precision

0.625

In [ ]:
combined = combined.merge(matches_rolling[["date", "team", "opponent", "result"]], left_index=True, right_index=True)

In [ ]:
combined

,actual,predicted,date,team,opponent,result
54,0,0,2022-01-01,Arsenal,Manchester City,L
55,0,0,2022-01-23,Arsenal,Burnley,D
56,1,0,2022-02-10,Arsenal,Wolves,W
57,1,0,2022-02-19,Arsenal,Brentford,W
58,1,1,2022-02-24,Arsenal,Wolves,W
...,...,...,...,...,...,...
1312,1,0,2022-03-13,Wolverhampton Wanderers,Everton,W
1313,0,0,2022-03-18,Wolverhampton Wanderers,Leeds United,L
1314,1,0,2022-04-02,Wolverhampton Wanderers,Aston Villa,W
1315,0,0,2022-04-08,Wolverhampton Wanderers,Newcastle Utd,L


In [ ]:
class MissingDict(dict):
  __missing__ = lambda self, key: key
map_values = {"Brighton and Hove Albion": "Brighton",
              "Manchester United": "Manchester",
              "Newcastle United": "Newcastle",
              "Tottenham Hotspur": "Tottenham",
              "West Ham United": "West Ham"
              ,"Wolverhampton Wanderers" : "Wolves"}
mapping = MissingDict(**map_values)

In [ ]:
combined["new_team"] = combined["team"].map(mapping)

In [ ]:
merged = combined.merge(combined, left_on=["date", "new_team"], right_on=["date", "opponent"])

In [ ]:
merged[(merged["predicted_x"] == 1 ) & merged["predicted_y"] ==0]["actual_x"].value_counts()

,count
actual_x,
0,144
1,88


In [ ]:
144/(144+68)

0.6792452830188679